
# 💰 Project 7 — RouteIQ: Cost-Aware Agent Router

**Core Concept:** Route requests to the right model based on complexity to minimize cost

### Architecture
Request → Complexity Score → Model Selection → Response + Cost Report
Simple  → Small Model  (cheap)
Medium  → Medium Model (balanced)
Complex → Large Model  (expensive)


Install

In [2]:
!pip install -q langchain langchain-groq langchain-core loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.5 MB/s eta 0:00:00


API Key

In [1]:
import os
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

All Setup In One Block

In [8]:
import os
import time
import json
from datetime import datetime, timezone
from loguru import logger
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import sys

logger.remove()
logger.add(sys.stdout, format="{time:HH:mm:ss} | {level} | {message}", level="DEBUG")

# ── Model Registry ───────────────────────────────────────────
MODEL_REGISTRY = {
    "small": {
        "model_id": "llama-3.1-8b-instant",
        "display_name": "LLaMA 3.1 8B",
        "cost_per_1k_tokens": 0.0001,
        "max_tokens": 1024,
        "best_for": "simple questions, FAQs, basic lookups",
        "complexity_range": (0, 3)
    },
    "medium": {
        "model_id": "llama-3.3-70b-versatile",
        "display_name": "LLaMA 3.3 70B",
        "cost_per_1k_tokens": 0.0008,
        "max_tokens": 2048,
        "best_for": "analysis, summaries, moderate reasoning",
        "complexity_range": (3, 6)
    },
    "large": {
        "model_id": "llama-3.3-70b-versatile",
        "display_name": "LLaMA 3.3 70B (High Temp)",
        "cost_per_1k_tokens": 0.0015,
        "max_tokens": 4096,
        "best_for": "complex reasoning, coding, research",
        "complexity_range": (6, 10)
    }
}

# ── Complexity Scorer ────────────────────────────────────────
class ComplexityScorer:
    def __init__(self):
        self.simple_keywords = [
            "what is", "who is", "when", "where", "define",
            "meaning of", "tell me", "simple", "basic", "quick"
        ]
        self.medium_keywords = [
            "explain", "compare", "analyze", "summarize", "describe",
            "how does", "difference between", "pros and cons", "review"
        ]
        self.complex_keywords = [
            "implement", "build", "create", "develop", "architect",
            "design", "optimize", "debug", "code", "algorithm",
            "research", "comprehensive", "detailed", "advanced"
        ]

    def score(self, request: str) -> dict:
        request_lower = request.lower()
        score = 5.0

        for keyword in self.simple_keywords:
            if keyword in request_lower:
                score -= 1.5

        for keyword in self.medium_keywords:
            if keyword in request_lower:
                score += 0.5

        for keyword in self.complex_keywords:
            if keyword in request_lower:
                score += 1.5

        word_count = len(request.split())
        if word_count > 50:
            score += 1.0
        elif word_count < 10:
            score -= 1.0

        if "?" in request and len(request) < 50:
            score -= 0.5

        score = max(0, min(10, score))

        if score < 3:
            tier = "small"
        elif score < 6:
            tier = "medium"
        else:
            tier = "large" # Changed from "complex" to "large"

        return {
            "score": round(score, 1),
            "tier": tier,
            "word_count": word_count
        }

# ── Cost Tracker ─────────────────────────────────────────────
class CostTracker:
    def __init__(self):
        self.records = []
        self.total_cost = 0.0
        self.total_tokens = 0

    def log(self, request: str, model_tier: str, model_id: str,
            tokens_used: int, latency_ms: float):
        cost = (tokens_used / 1000) * MODEL_REGISTRY[model_tier]["cost_per_1k_tokens"]
        self.total_cost += cost
        self.total_tokens += tokens_used
        self.records.append({
            "request": request[:50],
            "model_tier": model_tier,
            "model_id": model_id,
            "tokens_used": tokens_used,
            "cost": round(cost, 6),
            "latency_ms": round(latency_ms, 2),
            "timestamp": datetime.now(timezone.utc).isoformat()
        })
        logger.info(f"Cost tracked: {model_tier} | Tokens: {tokens_used} | Cost: ${cost:.6f}")

    def get_report(self) -> dict:
        if not self.records:
            return {"total_cost": 0, "total_tokens": 0, "records": []}

        by_tier = {}
        for record in self.records:
            tier = record["model_tier"]
            if tier not in by_tier:
                by_tier[tier] = {"calls": 0, "cost": 0, "tokens": 0}
            by_tier[tier]["calls"] += 1
            by_tier[tier]["cost"] += record["cost"]
            by_tier[tier]["tokens"] += record["tokens_used"]

        return {
            "total_cost": round(self.total_cost, 6),
            "total_tokens": self.total_tokens,
            "total_requests": len(self.records),
            "by_tier": by_tier,
            "records": self.records
        }

    def calculate_savings(self) -> dict:
        if not self.records:
            return {"savings": 0, "percentage": 0}

        large_cost_per_token = MODEL_REGISTRY["large"]["cost_per_1k_tokens"] / 1000
        cost_if_all_large = sum(
            r["tokens_used"] * large_cost_per_token
            for r in self.records
        )
        actual_cost = self.total_cost
        savings = cost_if_all_large - actual_cost
        percentage = (savings / cost_if_all_large * 100) if cost_if_all_large > 0 else 0

        return {
            "cost_if_all_large": round(cost_if_all_large, 6),
            "actual_cost": round(actual_cost, 6),
            "savings": round(savings, 6),
            "percentage_saved": round(percentage, 1)
        }

# ── RouteIQ Agent ────────────────────────────────────────────
class RouteIQAgent:
    def __init__(self):
        self.scorer = ComplexityScorer()
        self.cost_tracker = CostTracker()
        self.models = {}

        for tier, config in MODEL_REGISTRY.items():
            temperature = 0.7 if tier == "large" else 0.3
            self.models[tier] = ChatGroq(
                model=config["model_id"],
                temperature=temperature,
                max_tokens=config["max_tokens"],
                api_key=os.environ["GROQ_API_KEY"]
            )

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful AI assistant. Answer the user's question clearly and concisely."),
            ("human", "{request}")
        ])

        logger.info("RouteIQ agent initialized with 3 model tiers")

    def route(self, request: str) -> dict:
        logger.info(f"Routing: {request[:50]}")

        complexity = self.scorer.score(request)
        tier = complexity["tier"]
        model_config = MODEL_REGISTRY[tier]

        logger.info(f"Complexity score: {complexity['score']} | Tier: {tier} | Model: {model_config['display_name']}")

        llm = self.models[tier]
        chain = self.prompt | llm

        start_time = time.time()
        response = chain.invoke({"request": request})
        latency_ms = (time.time() - start_time) * 1000

        answer = response.content
        tokens_used = len(request.split()) + len(answer.split())

        self.cost_tracker.log(
            request, tier,
            model_config["model_id"],
            tokens_used, latency_ms
        )

        return {
            "request": request,
            "complexity_score": complexity["score"],
            "complexity_tier": tier,
            "model_used": model_config["display_name"],
            "model_id": model_config["model_id"],
            "answer": answer,
            "tokens_used": tokens_used,
            "latency_ms": round(latency_ms, 2),
            "cost": round((tokens_used / 1000) * model_config["cost_per_1k_tokens"], 6)
        }

    def display_result(self, result: dict):
        print("\n" + "="*55)
        print("ROUTEIQ RESULT")
        print("="*55)
        print(f"Request         : {result['request'][:60]}")
        print(f"Complexity Score: {result['complexity_score']}/10")
        print(f"Complexity Tier : {result['complexity_tier'].upper()}")
        print(f"Model Used      : {result['model_used']}")
        print(f"Tokens Used     : {result['tokens_used']}")
        print(f"Latency         : {result['latency_ms']}ms")
        print(f"Cost            : ${result['cost']}")
        print(f"\nAnswer: {result['answer'][:300]}")
        print("="*55)

agent = RouteIQAgent()
print("RouteIQ agent ready")

13:22:46 | INFO | RouteIQ agent initialized with 3 model tiers
RouteIQ agent ready


Test Simple Requests

In [6]:
print("========== SIMPLE REQUESTS ==========\n")

simple_requests = [
    "What is Python?",
    "Who invented the telephone?",
    "What is the capital of France?"
]

for request in simple_requests:
    result = agent.route(request)
    agent.display_result(result)

========== SIMPLE REQUESTS ==========

13:21:53 | INFO | Routing: What is Python?
13:21:53 | INFO | Complexity score: 2.0 | Tier: small | Model: LLaMA 3.1 8B
13:21:54 | INFO | Cost tracked: small | Tokens: 168 | Cost: $0.000017

ROUTEIQ RESULT
Request         : What is Python?
Complexity Score: 2.0/10
Complexity Tier : SMALL
Model Used      : LLaMA 3.1 8B
Tokens Used     : 168
Latency         : 592.14ms
Cost            : $1.7e-05

Answer: **What is Python?**

Python is a high-level, interpreted programming language that is widely used for various purposes such as:

* **Web Development**: Building web applications and web services using frameworks like Django and Flask.
* **Data Analysis**: Working with data using libraries like Panda
13:21:54 | INFO | Routing: Who invented the telephone?
13:21:54 | INFO | Complexity score: 3.5 | Tier: medium | Model: LLaMA 3.3 70B
13:21:54 | INFO | Cost tracked: medium | Tokens: 75 | Cost: $0.000060

ROUTEIQ RESULT
Request         : Who invented the te

Test Medium Requests

In [9]:
print("========== MEDIUM REQUESTS ==========\n")

medium_requests = [
    "Compare REST APIs and GraphQL and explain the pros and cons of each",
    "Summarize the key differences between supervised and unsupervised learning",
    "Explain how transformer architecture works in large language models"
]

for request in medium_requests:
    result = agent.route(request)
    agent.display_result(result)

========== MEDIUM REQUESTS ==========

13:22:58 | INFO | Routing: Compare REST APIs and GraphQL and explain the pros
13:22:58 | INFO | Complexity score: 6.5 | Tier: large | Model: LLaMA 3.3 70B (High Temp)
13:23:00 | INFO | Cost tracked: large | Tokens: 562 | Cost: $0.000843

ROUTEIQ RESULT
Request         : Compare REST APIs and GraphQL and explain the pros and cons 
Complexity Score: 6.5/10
Complexity Tier : LARGE
Model Used      : LLaMA 3.3 70B (High Temp)
Tokens Used     : 562
Latency         : 2662.38ms
Cost            : $0.000843

Answer: **Introduction to REST APIs and GraphQL**

REST (Representational State of Resource) APIs and GraphQL are two popular approaches to building web APIs. While both enable data exchange between clients and servers, they differ significantly in their design principles, advantages, and use cases.

**REST
13:23:00 | INFO | Routing: Summarize the key differences between supervised a
13:23:00 | INFO | Complexity score: 4.5 | Tier: medium | Model: LLaMA 

In [ ]:
Test Complex Requests

In [10]:
print("========== COMPLEX REQUESTS ==========\n")

complex_requests = [
    "Implement a binary search tree in Python with insert, delete, and search methods",
    "Design a comprehensive microservices architecture for an e-commerce platform with detailed component breakdown",
    "Build a detailed algorithm for optimizing database query performance with indexing strategies"
]

for request in complex_requests:
    result = agent.route(request)
    agent.display_result(result)

========== COMPLEX REQUESTS ==========

13:23:28 | INFO | Routing: Implement a binary search tree in Python with inse
13:23:28 | INFO | Complexity score: 6.5 | Tier: large | Model: LLaMA 3.3 70B (High Temp)
13:23:30 | INFO | Cost tracked: large | Tokens: 553 | Cost: $0.000830

ROUTEIQ RESULT
Request         : Implement a binary search tree in Python with insert, delete
Complexity Score: 6.5/10
Complexity Tier : LARGE
Model Used      : LLaMA 3.3 70B (High Temp)
Tokens Used     : 553
Latency         : 2532.22ms
Cost            : $0.00083

Answer: ## Binary Search Tree Implementation in Python
### Overview

This implementation includes a `Node` class that represents each node in the binary search tree, and a `BinarySearchTree` class that provides methods for inserting, deleting, and searching nodes.

### Code

```python
class Node:
    """Rep
13:23:30 | INFO | Routing: Design a comprehensive microservices architecture 
13:23:30 | INFO | Complexity score: 10 | Tier: large | Model: LLaMA 3.

Cost Report

In [11]:
print("========== COST REPORT ==========\n")
report = agent.cost_tracker.get_report()
savings = agent.cost_tracker.calculate_savings()

print(f"Total Requests  : {report['total_requests']}")
print(f"Total Tokens    : {report['total_tokens']}")
print(f"Total Cost      : ${report['total_cost']}")

print(f"\nBy Model Tier:")
for tier, data in report['by_tier'].items():
    print(f"  {tier:8} : {data['calls']} calls | {data['tokens']} tokens | ${round(data['cost'], 6)}")

print(f"\nCost Savings Analysis:")
print(f"  Cost if all large model : ${savings['cost_if_all_large']}")
print(f"  Actual cost             : ${savings['actual_cost']}")
print(f"  Savings                 : ${savings['savings']}")
print(f"  Percentage saved        : {savings['percentage_saved']}%")

========== COST REPORT ==========

Total Requests  : 6
Total Tokens    : 2854
Total Cost      : $0.004176

By Model Tier:
  large    : 5 calls | 2704 tokens | $0.004056
  medium   : 1 calls | 150 tokens | $0.00012

Cost Savings Analysis:
  Cost if all large model : $0.004281
  Actual cost             : $0.004176
  Savings                 : $0.000105
  Percentage saved        : 2.5%


Project Summary

In [12]:
print("========== ROUTEIQ SUMMARY ==========\n")
print("Project      : RouteIQ — Cost-Aware Agent Router")
print("Author       : K Murali Krishna")
print("\nModel Tiers:")
for tier, config in MODEL_REGISTRY.items():
    print(f"  {tier:8} : {config['display_name']:30} | Cost: ${config['cost_per_1k_tokens']}/1k tokens")
print("\nKey Capabilities:")
print("  ✓ Keyword-based complexity scoring")
print("  ✓ Automatic model tier selection")
print("  ✓ Per-request cost tracking")
print("  ✓ Savings calculation vs all-large baseline")
print("  ✓ Latency tracking per model")
print("\nProduction Concepts Demonstrated:")
print("  ✓ Cost optimization through intelligent routing")
print("  ✓ Model registry pattern")
print("  ✓ Complexity classification")
print("  ✓ Token-based cost calculation")

========== ROUTEIQ SUMMARY ==========

Project      : RouteIQ — Cost-Aware Agent Router
Author       : K Murali Krishna

Model Tiers:
  small    : LLaMA 3.1 8B                   | Cost: $0.0001/1k tokens
  medium   : LLaMA 3.3 70B                  | Cost: $0.0008/1k tokens
  large    : LLaMA 3.3 70B (High Temp)      | Cost: $0.0015/1k tokens

Key Capabilities:
  ✓ Keyword-based complexity scoring
  ✓ Automatic model tier selection
  ✓ Per-request cost tracking
  ✓ Savings calculation vs all-large baseline
  ✓ Latency tracking per model

Production Concepts Demonstrated:
  ✓ Cost optimization through intelligent routing
  ✓ Model registry pattern
  ✓ Complexity classification
  ✓ Token-based cost calculation
